# 📚 Smart Research Assistant — RAG-based Knowledge System

**Intern:** Priyanshi  
**Program:** Celebal Excellence Data Science Internship 2026  
**Project:** Smart Research Assistant (Retrieval-Augmented Generation)  
---

## Objective

Demonstrate an end-to-end Retrieval-Augmented Generation (RAG) pipeline that allows a user to upload documents and ask natural-language questions, with answers grounded in retrieved context and scored for reliability.

## Pipeline

1. **Data Ingestion** — load and chunk documents
2. **Storage** — embed chunks and store in a vector database (FAISS)
3. **Query Processing** — embed the user's question
4. **Answer Generation** — retrieve relevant chunks, generate a grounded answer
5. **Evaluation** — score the answer for faithfulness and relevance

This notebook mirrors the production `src/` modules in this repo (`ingest.py`, `vectorstore.py`, `rag_chain.py`, `evaluate.py`) so the logic here matches the Streamlit app exactly.

## 1. Setup

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.ingest import load_document, chunk_documents
from src.vectorstore import build_vectorstore, load_vectorstore
from src.rag_chain import build_rag_chain, ask
from src.evaluate import evaluate_answer

from dotenv import load_dotenv
load_dotenv("../.env")

USER_AGENT environment variable not set, consider setting it to identify your requests.


False

## 2. Data Ingestion

Place a sample PDF (e.g. an research paper) inside `data/sample_docs/` before running this cell.

In [2]:
from pathlib import Path

sample_dir = Path("../data/sample_docs")
pdf_files = list(sample_dir.glob("*.pdf"))
print(f"Found {len(pdf_files)} sample PDF(s): {[p.name for p in pdf_files]}")

assert pdf_files, "Add at least one PDF to data/sample_docs/ before continuing."

raw_docs = load_document(str(pdf_files[0]))
print(f"Loaded {len(raw_docs)} page(s) from {pdf_files[0].name}")

Found 1 sample PDF(s): ['Week7_Project.pdf']
Loaded 4 page(s) from Week7_Project.pdf


In [3]:
chunks = chunk_documents(raw_docs, chunk_size=1000, chunk_overlap=150)
print(f"Split into {len(chunks)} chunks.\n")
print("Sample chunk:\n", chunks[0].page_content[:400])

Split into 6 chunks.

Sample chunk:
 Document  Question  Answering  System  (RAG)  
Dataset:  
Simple  Beginner  Dataset  (Easiest)  
Use  your  own  PDFs:  
●  Notes  ●  Resume  ●  Research  papers  ●  Books  ●  RAG  is  meant  for  custom/private  data  
Or  Try  this  Hugging  Face  Dataset 
Reference:Github  Link   
Overview  
This  project  implements  a  Retrieval-Augmented  Generation  (RAG)  system  that  
answers
 
questions


## 3. Storage — Embed + Build Vector Index

Using a free Hugging Face sentence-transformer for embeddings (`all-MiniLM-L6-v2`), stored in FAISS for fast local similarity search.

In [4]:
vectorstore = build_vectorstore(chunks, backend="faiss")
print("FAISS index built and saved to data/faiss_index/")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index built and saved to data/faiss_index/


## 4. Query Processing + Answer Generation

`LLM_BACKEND=huggingface` runs fully offline with no API key. Switch to `"openai"` in `.env` once a key is available for higher-quality answers.

In [5]:
llm_backend = os.getenv("LLM_BACKEND", "huggingface")
chain = build_rag_chain(vectorstore, backend=llm_backend, k=4)

question = "What is the leave policy?"  # replace with a question relevant to your sample doc
result = ask(chain, question)

print("Q:", question)
print("\nA:", result["answer"])
print(f"\nRetrieved {len(result['sources'])} supporting chunk(s).")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Q: What is the leave policy?

A: Text is encapsulated into a language model.

Retrieved 4 supporting chunk(s).


### Supporting document context

Showing the retrieved chunks that grounded the answer above — this is what the UI surfaces as "source-backed responses".

In [6]:
for i, doc in enumerate(result["sources"], 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300], "...\n")

--- Chunk 1 ---
The
 
system
 
retrieves
 
the
 
most
 
relevant
 
chunks
 
from
 
the
 
database.
 7.  Answer  Generation  
A
 
language
 
model
 
generates
 
an
 
answer
 
using
 
the
 
retrieved
 
context.
 
Data  
Input  sources  include:  
●  PDF  documents  ●  Text  files  ●  Notes  or  articles  
These  are  ...

--- Chunk 2 ---
This
 
improves
 
factual
 
accuracy
 
and
 
allows
 
question
 
answering
 
over
 
private
 
or
 
domain-specific
 
data.
 
Objectives  
●  Understand  the  concept  of  Retrieval-Augmented  Generation  (RAG)  ●  Build  a  pipeline  combining  retrieval  and  generation  ●  Enable  question  answer ...

--- Chunk 3 ---
Document  Question  Answering  System  (RAG)  
Dataset:  
Simple  Beginner  Dataset  (Easiest)  
Use  your  own  PDFs:  
●  Notes  ●  Resume  ●  Research  papers  ●  Books  ●  RAG  is  meant  for  custom/private  data  
Or  Try  this  Hugging  Face  Dataset 
Reference:Github  Link   
Overview  
This ...

--- Chunk 4 ---
3.  Generation  
A

## 5. Evaluation

Scores the answer for reliability. Uses RAGAs (faithfulness, answer relevancy, context precision) when `OPENAI_API_KEY` is set; otherwise falls back to a lexical-overlap heuristic so evaluation still runs offline.

In [7]:
contexts = [doc.page_content for doc in result["sources"]]
scores = evaluate_answer(question, result["answer"], contexts)
scores

{'lexical_overlap_score': 0.667,
 'note': 'RAGAs skipped (no OPENAI_API_KEY) — showing offline heuristic instead.'}

## 6. Try More Questions

Loop through a few sample questions to sanity-check the pipeline end-to-end before wiring it into the Streamlit app (`app.py`).

In [8]:
sample_questions = [
    "Summarize this document in 3 sentences.",
    "What are the key points covered?",
]

for q in sample_questions:
    r = ask(chain, q)
    s = evaluate_answer(q, r["answer"], [d.page_content for d in r["sources"]])
    print(f"Q: {q}\nA: {r['answer']}\nScore: {s}\n{'-'*60}")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Q: Summarize this document in 3 sentences.
A: Query Processing
Score: {'lexical_overlap_score': 1.0, 'note': 'RAGAs skipped (no OPENAI_API_KEY) — showing offline heuristic instead.'}
------------------------------------------------------------


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Q: What are the key points covered?
A: A language model.
Score: {'lexical_overlap_score': 1.0, 'note': 'RAGAs skipped (no OPENAI_API_KEY) — showing offline heuristic instead.'}
------------------------------------------------------------


## Learning Outcomes

- How LLMs integrate with external knowledge via retrieval
- Working of vector databases (FAISS) and dense embeddings
- Prompt engineering for grounded, context-restricted answers
- Retrieval-based AI system design (chunking, top-k retrieval, `stuff` chain)
- Importance of evaluation (RAGAs) in judging AI system reliability

## Next Steps / Future Enhancements

- [ ] Swap in ChromaDB backend for persistent storage (`build_vectorstore(chunks, backend="chroma")`)
- [ ] Add multi-turn conversational memory
- [ ] Deploy Streamlit app to Streamlit Community Cloud / AWS
- [ ] Add multi-language support